In [ ]:
import pandas as pd
import pycountry
import wbdata

In [ ]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
            "prepare_wacc",
    )

### File paths

In [ ]:
wacc_fn = snakemake.input.wacc
bus_locations_fn = snakemake.input.bus_locations

# Output
wacc_clustered_fn = snakemake.output.wacc

# config
config = snakemake.config

### Get WACC data

In [ ]:
# Read the CSV data
wacc = pd.read_csv(wacc_fn, comment='#', index_col="iso_alpha")


### Cluster per region


Map each country in the production data to its region using the region definitions from the config file.


In [ ]:
# Load region definitions from config file
regions = config['regions']

In [ ]:
country_name_corrections = {
    "Democratic Republic of the Congo": "Congo, The Democratic Republic of the",
    "Republic of the Congo": "Republic of the Congo",
    "Kosovo": "Republic of Kosovo",  # pycountry not supported
    "Russia": "Russian Federation",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Tanzania": "United Republic of Tanzania",
    "Bolivia": "Plurinational State of Bolivia",
    "Vietnam": "Viet Nam",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Taiwan": "Taiwan, Province of China",
    "Laos": "Lao People's Democratic Republic",
    "Brunei": "Brunei Darussalam",
    "Equatorial French Guiana": "French Guiana",
    "Syria": "Syrian Arab Republic",   
    "Palestine": "Palestine, State of",
    "Moldova": "Republic of Moldova",
    "North Korea": "Korea, Democratic People's Republic of",
}

In [ ]:
# Build a mapping from country name to ISO-3 code (still needed for GDP World Bank lookup)
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_3
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_3

# Build a mapping from ISO-3 code to region (config regions now use ISO-3 codes directly)
iso_to_region = {}
for region, iso_codes in regions.items():
    for code in iso_codes:
        iso_to_region[code] = region

In [ ]:
# Map each row in production to its region
wacc['region'] = wacc.index.map(lambda iso: iso_to_region.get(iso, 'Other'))

### 

### Cluster: Weighted by GDP

In [ ]:
# --- fetch GDP (current USD) from World Bank ---
gdp_raw = wbdata.get_dataframe(
    {"NY.GDP.MKTP.CD": "gdp_usd"},
    date="2024",
)
gdp = gdp_raw.dropna(subset=["gdp_usd"]).reset_index()
gdp = gdp.rename(columns={"country": "country_name"})

# map country name → ISO-3 so we can merge on the wacc index
gdp["country_code"] = gdp["country_name"].map(
    lambda name: country_name_to_iso.get(name)
)
gdp = gdp.dropna(subset=["country_code"])

# --- attach GDP weights ---
wacc_gdp = wacc.merge(gdp[["country_code", "gdp_usd"]], left_index=True, right_on="country_code", how="left")

# fill missing GDP with regional median so no country is dropped
wacc_gdp["gdp_usd"] = wacc_gdp.groupby("region")["gdp_usd"].transform(
    lambda x: x.fillna(x.median())
)

def weighted_mean(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

wacc_by_region = (
    wacc_gdp.groupby("region")
    .apply(lambda g: pd.Series({
        "wacc":                  weighted_mean(g, "wacc", "gdp_usd"),
        "wacc_real":             weighted_mean(g, "wacc_real", "gdp_usd"),
        "risk_free_rate":        weighted_mean(g, "risk_free_rate", "gdp_usd"),
        "country_risk_premium":  weighted_mean(g, "country_risk_premium", "gdp_usd"),
        "equity_risk_premium":   weighted_mean(g, "equity_risk_premium", "gdp_usd"),
        "cost_of_equity":        weighted_mean(g, "cost_of_equity", "gdp_usd"),
        "cost_of_debt":          weighted_mean(g, "cost_of_debt", "gdp_usd"),
        "n_countries":           len(g),
        "wacc_spread":           g["wacc"].max() - g["wacc"].min(),  # diagnostics
    }))
    .reset_index()
)

In [ ]:
wacc_by_region.set_index("region", inplace=True)

### Save clustered WACC

In [ ]:
wacc_by_region

In [ ]:
# Filter for relevant columns
wacc_by_region = wacc_by_region[["wacc_real", "n_countries", "wacc_spread"]]

In [ ]:
wacc_by_region.to_csv(wacc_clustered_fn)

### Mean (depreciated)

In [ ]:
# wacc_by_region = wacc.groupby('region').mean(numeric_only=True)
# wacc_by_region.round(3)